# Import Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Load Dataset

In [2]:
df = pd.read_csv("vgsales.csv")
df.head()

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
3,4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37


# Explore Dataset

In [3]:
print("Dataset Shape:", df.shape)

df.info()

df.describe()

Dataset Shape: (16598, 11)
<class 'pandas.DataFrame'>
RangeIndex: 16598 entries, 0 to 16597
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Rank          16598 non-null  int64  
 1   Name          16598 non-null  str    
 2   Platform      16598 non-null  str    
 3   Year          16327 non-null  float64
 4   Genre         16598 non-null  str    
 5   Publisher     16540 non-null  str    
 6   NA_Sales      16598 non-null  float64
 7   EU_Sales      16598 non-null  float64
 8   JP_Sales      16598 non-null  float64
 9   Other_Sales   16598 non-null  float64
 10  Global_Sales  16598 non-null  float64
dtypes: float64(6), int64(1), str(4)
memory usage: 1.4 MB


,Rank,Year,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
count,16598.000000,16327.000000,16598.000000,16598.000000,16598.000000,16598.000000,16598.000000
mean,8300.605254,2006.406443,0.264667,0.146652,0.077782,0.048063,0.537441
std,4791.853933,5.828981,0.816683,0.505351,0.309291,0.188588,1.555028
min,1.000000,1980.000000,0.000000,0.000000,0.000000,0.000000,0.010000
25%,4151.250000,2003.000000,0.000000,0.000000,0.000000,0.000000,0.060000
50%,8300.500000,2007.000000,0.080000,0.020000,0.000000,0.010000,0.170000
75%,12449.750000,2010.000000,0.240000,0.110000,0.040000,0.040000,0.470000
max,16600.000000,2020.000000,41.490000,29.020000,10.220000,10.570000,82.740000


# Check Missing Values

In [4]:
df.isnull().sum()

Rank              0
Name              0
Platform          0
Year            271
Genre             0
Publisher        58
NA_Sales          0
EU_Sales          0
JP_Sales          0
Other_Sales       0
Global_Sales      0
dtype: int64

# Select Required Columns

In [5]:
data = df[['Publisher',
           'Platform',
           'Genre',
           'Year',
           'Global_Sales']]

data.head()

,Publisher,Platform,Genre,Year,Global_Sales
0,Nintendo,Wii,Sports,2006.0,82.74
1,Nintendo,NES,Platform,1985.0,40.24
2,Nintendo,Wii,Racing,2008.0,35.82
3,Nintendo,Wii,Sports,2009.0,33.00
4,Nintendo,GB,Role-Playing,1996.0,31.37


# Remove Missing Target Values

In [6]:
data = data.dropna(subset=['Global_Sales'])

# Fill Missing Values

In [7]:
data['Publisher'] = data['Publisher'].fillna("Unknown")
data['Platform'] = data['Platform'].fillna("Unknown")
data['Genre'] = data['Genre'].fillna("Unknown")

data['Year'] = data['Year'].fillna(data['Year'].median())

# Features and Target

In [8]:
X = data[['Publisher',
          'Platform',
          'Genre',
          'Year']]

y = data['Global_Sales']

# Train Test Split

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Preprocessing

In [10]:
categorical_features = [
    'Publisher',
    'Platform',
    'Genre'
]

numeric_features = [
    'Year'
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            'cat',
            OneHotEncoder(handle_unknown='ignore'),
            categorical_features
        ),
        (
            'num',
            SimpleImputer(strategy='median'),
            numeric_features
        )
    ]
)

# Create Model Pipeline

In [11]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ))
])

# Train Model

In [12]:
model.fit(X_train, y_train)

Bad pipe message: %s [b'0.9,*/*;q=0.8\r\nHost: localhost:44337\r\nUser-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:152.0) Gecko/2010', b'01 Firefox/152.0\r\nAccept-Encoding: gzip, deflat', b' br, zstd\r\nAccept-Language: en-US,en;q=0.9\r\nReferer: https://obscure-trout-gw546x4jx4vfwg4j.github.d']
Bad pipe message: %s [b'/\r\nX-Request-ID: adb263d03ee8a46122f0d6bfb6993046\r\nX-Real-IP: 49.37.201.113\r\nX-Forwarded-Port: 443\r\n']


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](4,)","['Publisher','Platform','Genre','Year']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,4
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the outpu

# Predict

In [13]:
predictions = model.predict(X_test)

predictions[:10]

array([0.06639167, 0.41112863, 0.065525  , 0.04242861, 0.73761243,
       0.80573667, 0.99025762, 0.10234583, 0.2012575 , 0.1363825 ])

# Evaluate Model

In [14]:
mae = mean_absolute_error(y_test, predictions)

mse = mean_squared_error(y_test, predictions)

rmse = np.sqrt(mse)

r2 = r2_score(y_test, predictions)

print("Mean Absolute Error :", mae)
print("Mean Squared Error  :", mse)
print("Root Mean Squared Error :", rmse)
print("R2 Score :", r2)

Mean Absolute Error : 0.5320698628865437
Mean Squared Error  : 4.07861094762278
Root Mean Squared Error : 2.0195571167022686
R2 Score : 0.029217423082910532


# Actual vs Predicted

In [15]:
comparison = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': predictions
})

comparison.head(20)

,Actual,Predicted
0,0.15,0.066392
1,0.40,0.411129
2,0.02,0.065525
3,0.03,0.042429
4,0.36,0.737612
5,2.24,0.805737
6,0.39,0.990258
7,0.65,0.102346
8,0.21,0.201258
9,0.44,0.136383


# Predict New Game Sales

In [16]:
new_game = pd.DataFrame({
    'Publisher': ['Nintendo'],
    'Platform': ['Wii'],
    'Genre': ['Sports'],
    'Year': [2008]
})

prediction = model.predict(new_game)

print("Predicted Global Sales:", prediction[0], "Million Copies")

Predicted Global Sales: 2.78596013888889 Million Copies


# Predict Multiple Games

In [17]:
games = pd.DataFrame({
    'Publisher': [
        'Nintendo',
        'Ubisoft',
        'Electronic Arts',
        'Activision'
    ],
    'Platform': [
        'Wii',
        'PS4',
        'PC',
        'X360'
    ],
    'Genre': [
        'Sports',
        'Action',
        'Simulation',
        'Shooter'
    ],
    'Year': [
        2008,
        2018,
        2016,
        2012
    ]
})

games['Predicted_Global_Sales'] = model.predict(games)

games

,Publisher,Platform,Genre,Year,Predicted_Global_Sales
0,Nintendo,Wii,Sports,2008,2.785960
1,Ubisoft,PS4,Action,2018,0.621950
2,Electronic Arts,PC,Simulation,2016,0.275751
3,Activision,X360,Shooter,2012,9.304386


# Feature Importance (Optional)

In [18]:
feature_names = model.named_steps['preprocessor'].get_feature_names_out()

importances = model.named_steps['regressor'].feature_importances_

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
})

importance_df.sort_values(by='Importance', ascending=False).head(20)

,Feature,Importance
578,num__Year,0.421677
335,cat__Publisher_Nintendo,0.097022
573,cat__Genre_Role-Playing,0.046529
574,cat__Genre_Shooter,0.036600
570,cat__Genre_Platform,0.031344
540,cat__Platform_GB,0.028579
566,cat__Genre_Action,0.028328
539,cat__Platform_DS,0.025408
569,cat__Genre_Misc,0.022379
572,cat__Genre_Racing,0.022345


# Gradio Framework

In [ ]:
def predict_sales(publisher, platform, genre, year):

    sample = pd.DataFrame({
        "Publisher":[publisher],
        "Platform":[platform],
        "Genre":[genre],
        "Year":[year]
    })

    prediction = model.predict(sample)[0]
    return f"{prediction:.2f} Million Units"

In [21]:
!pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.0/31.0 MB 46.2 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.3/770.3 kB 32.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 38.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 38.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 38.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.6/676.6 kB 20.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31/31 [gradio]30/31 [gradio]face-hub]

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [22]:
import gradio as gr

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [23]:
demo = gr.Interface(
    fn=predict_sales,

    inputs=[
        gr.Dropdown(
            publishers,
            label="Publisher"
        ),

        gr.Dropdown(
            platforms,
            label="Platform"
        ),

        gr.Dropdown(
            genres,
            label="Genre"
        ),

        gr.Number(
            label="Release Year",
            value=2015
        )
    ],

    outputs=gr.Textbox(
        label="Predicted Global Sales"
    ),

    title="🎮 Video Game Global Sales Prediction",

    description="""
Predict Global Sales of a Video Game using

• Publisher

• Platform

• Genre

• Release Year

Model: Random Forest Regressor
"""
)

demo.launch()

NameError: name 'publishers' is not defined